# Mooring Field Detection — Kaggle GPU Training

**Settings (only two required):**
1. Accelerator → **GPU T4** (or P100)
2. Internet → **On**

You do **not** need to upload or attach any dataset — the training images and
corrected labels are already committed in the GitHub repo and come down with the clone.

Run the 3 code cells **in order**. Do not restart the kernel between cells.
Training takes a few hours; run it **interactively** (avoid Save & Run All timeouts),
then Save Version at the end to keep the output.

Package entry points: `runtime.bootstrap_kaggle` → `train_boats.train` → `runtime.publish_outputs`.

In [ ]:
# Cell 1 — Clone repo + install
import subprocess, sys, shutil, os
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
URL = "https://github.com/IshanKasam/MooringFieldDetection.git"

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "python-dotenv"], check=True)

import mooring_fields
print("mooring_fields:", mooring_fields.__file__)
print("cwd:", os.getcwd())
print("/kaggle/input:", sorted(p.name for p in Path("/kaggle/input").iterdir()) if Path("/kaggle/input").exists() else None)

In [ ]:
# Cell 2 — Bootstrap data + GPU (src/mooring_fields/runtime.py)
import sys, json, os
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

from mooring_fields.runtime import bootstrap_kaggle

info = bootstrap_kaggle()
print(json.dumps(info, indent=2))
assert info["cuda"] is True, "Enable GPU T4 in notebook Settings → Accelerator"
assert info["has_imagery"] and info["has_labels"], info
assert info["counts"]["imagery_train"] > 0 and info["counts"]["labels_train"] > 0, info["counts"]

In [ ]:
# Cell 3 — Train on corrected labels (src/mooring_fields/train_boats.py)
import sys, json, os
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

from mooring_fields.train_boats import train
from mooring_fields.runtime import publish_outputs

report = train(use_corrected_labels=True)
report["published"] = publish_outputs()
print(json.dumps({k: v for k, v in report.items() if k != "results"}, indent=2))

best = Path(report["best_weights"] or "")
assert best.is_file(), report
print("READY:", best)

## Download

**Save Version** → open version → **Output** → `mooring_outputs/mooring_boats/weights/best.pt`